# MediCore Monitoring Data Analysis

This notebook analyses monitoring data collected from the MediCore cloud infrastructure deployment.

The aim is to evaluate system performance, security events and storage growth trends. The results will be used to support operational decision making, risk management and compliance reporting.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme()

df = pd.read_csv("medicore_monitoring_data.csv")

df["timestamp"] = pd.to_datetime(df["timestamp"])

print(df.shape)
df.head()

: 

In [ ]:
cpu_df = (
    df[df["vm_id"].str.contains("medicore", case=False)]
    .groupby("timestamp")["cpu_usage_pct"]
    .mean()
    .reset_index()
)

cpu_df["cpu_7d_ma"] = (
    cpu_df["cpu_usage_pct"]
    .rolling(window=168, min_periods=1)
    .mean()
)

peak_row = cpu_df.loc[cpu_df["cpu_usage_pct"].idxmax()]

plt.figure(figsize=(12, 6))

plt.plot(
    cpu_df["timestamp"],
    cpu_df["cpu_usage_pct"],
    label="Average CPU Usage"
)

plt.plot(
    cpu_df["timestamp"],
    cpu_df["cpu_7d_ma"],
    label="7-Day Moving Average",
    linewidth=3
)

plt.scatter(
    peak_row["timestamp"],
    peak_row["cpu_usage_pct"],
    color="red"
)

plt.annotate(
    f'Peak = {peak_row["cpu_usage_pct"]:.1f}%',
    (peak_row["timestamp"], peak_row["cpu_usage_pct"])
)

plt.title("Figure 1 - Average CPU Usage Over Time")
plt.xlabel("Date")
plt.ylabel("CPU Usage (%)")
plt.legend()

plt.tight_layout()

plt.savefig("figure1-cpu-usage.png")

plt.show()

## Figure 1 Analysis

Figure 1 shows average CPU utilisation across the MediCore infrastructure over time.

The highest CPU usage reached approximately XX%, indicating periods of elevated demand. The 7-day moving average smooths short-term fluctuations and provides a clearer view of long-term utilisation trends.

The results suggest that the infrastructure is capable of supporting normal workloads while still experiencing predictable business-hour peaks. Where CPU utilisation approaches the scaling threshold, the configured Auto Scaling Group would help maintain service availability.

Based on the observed utilisation levels, the deployed AWS environment appears appropriately sized for the current MediCore workload of approximately 140,000 patient records. Continued monitoring is recommended to identify future growth trends and trigger capacity planning activities where necessary.